![mobydick](mobydick.jpg)

In this workspace, you'll scrape the novel Moby Dick from the website [Project Gutenberg](https://www.gutenberg.org/) (which contains a large corpus of books) using the Python `requests` package. You'll extract words from this web data using `BeautifulSoup` before analyzing the distribution of words using the Natural Language ToolKit (`nltk`) and `Counter`.

The Data Science pipeline you'll build in this workspace can be used to visualize the word frequency distributions of any novel you can find on Project Gutenberg.

In [10]:
# Import and download packages
import requests
from bs4 import BeautifulSoup
import nltk
from collections import Counter
nltk.download('stopwords')

# Start coding here... 

[nltk_data] Downloading package stopwords to /home/repl/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Step 2 — Request and encode the text

We send an HTTP GET request to the cached HTML version of Moby Dick and set the encoding to `utf-8` so characters are interpreted correctly.

In [11]:
# Fetch the Moby Dick HTML page
url = 'https://s3.amazonaws.com/assets.datacamp.com/production/project_147/datasets/2701-h.htm'
r = requests.get(url)

# Set the correct encoding
r.encoding = 'utf-8'

## Step 3 — Extract the HTML text

The `.text` attribute returns the full HTML source as a Python string. We print the first 2000 characters as a sanity check.

In [12]:
# Extract raw HTML as a string
html = r.text

# Preview the first 2000 characters
print(html[0:2000])

<?xml version="1.0" encoding="utf-8"?>

<!DOCTYPE html
   PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
   "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd" >

<html xmlns="http://www.w3.org/1999/xhtml" lang="en">
  <head>
    <title>
      Moby Dick; Or the Whale, by Herman Melville
    </title>
    <style type="text/css" xml:space="preserve">

    body { background:#faebd0; color:black; margin-left:15%; margin-right:15%; text-align:justify }
    P { text-indent: 1em; margin-top: .25em; margin-bottom: .25em; }
    H1,H2,H3,H4,H5,H6 { text-align: center; margin-left: 15%; margin-right: 15%; }
    hr  { width: 50%; text-align: center;}
    .foot { margin-left: 20%; margin-right: 20%; text-align: justify; text-indent: -3em; font-size: 90%; }
    blockquote {font-size: 100%; margin-left: 0%; margin-right: 0%;}
    .mynote    {background-color: #DDE; color: #000; padding: .5em; margin-left: 10%; margin-right: 10%; font-family: sans-serif; font-size: 95%;}
    .toc       { margin-left: 10%; m

## Step 4 — Parse HTML and extract plain text

We create a `BeautifulSoup` object using Python's built-in `html.parser`, then call `.get_text()` to strip away all HTML tags and return just the readable text.

In [13]:
# Create a BeautifulSoup object
html_soup = BeautifulSoup(html, 'html.parser')

# Extract plain text from the parsed HTML
moby_text = html_soup.get_text()

## Step 5 — Tokenize the text

We use NLTK's `RegexpTokenizer` with the pattern `\w+` to keep only alphanumeric words, automatically discarding punctuation and whitespace.

In [14]:
# Initialize the regex tokenizer — \w+ matches any alphanumeric sequence
tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+')

# Tokenize the full text
tokens = tokenizer.tokenize(moby_text)

print(f'Total tokens: {len(tokens)}')
print('First 10 tokens:', tokens[:10])

Total tokens: 222624
First 10 tokens: ['Moby', 'Dick', 'Or', 'the', 'Whale', 'by', 'Herman', 'Melville', 'The', 'Project']


## Step 6 — Convert to lowercase

Normalise all tokens to lowercase so `Whale` and `whale` are counted as the same word.

In [15]:
# Convert every token to lowercase
words = [word.lower() for word in tokens]

print('First 8 words:', words[:8])

First 8 words: ['moby', 'dick', 'or', 'the', 'whale', 'by', 'herman', 'melville']


## Step 7 — Load English stop words

Stop words are high-frequency function words ("the", "and", "of", ...) that carry little meaning on their own. NLTK ships with a curated list for English.

In [16]:
# Load the English stop word list
stop_words = nltk.corpus.stopwords.words('english')

print('First 8 stop words:', stop_words[:8])

First 8 stop words: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all']


## Step 8 — Remove stop words

We filter out any word that appears in the stop word list, leaving only content-bearing vocabulary.

In [17]:
# Keep only words that are NOT in the stop word list
words_no_stop = [word for word in words if word not in stop_words]

print(f'Words after removing stop words: {len(words_no_stop)}')
print('First 5 words:', words_no_stop[:5])

Words after removing stop words: 113214
First 5 words: ['moby', 'dick', 'whale', 'herman', 'melville']


## Step 9 — Count word frequencies

`Counter` tallies occurrences of each word. We then call `.most_common(10)` to retrieve the ten highest-frequency words and their counts.

In [18]:
# Build a frequency counter
count = Counter(words_no_stop)

# Get the ten most common words
top_ten = count.most_common(10)

print('Top 10 most frequent words in Moby Dick:')
print(f'{"Word":<15} {"Count"}')
print('-' * 25)
for word, freq in top_ten:
    print(f'{word:<15} {freq}')

Top 10 most frequent words in Moby Dick:
Word            Count
-------------------------
whale           1246
one             925
like            647
upon            568
man             527
ship            519
ahab            517
ye              473
sea             455
old             452


## Results

The output above shows the ten most meaningful words in *Moby Dick* after stripping away grammatical filler. Words like **whale**, **ship**, and **sea** dominate — entirely consistent with the novel's themes of obsession and the ocean. The high frequency of **old** and **man** echoes the characterisation of Captain Ahab and the seafaring world Melville depicts.